In [ ]:
# Part 1: Create a Kafka Topic
# Open a terminal in JupyterLab (File > New > Terminal) and create a topic:

# /usr/local/kafka/bin/kafka-topics.sh --create \
#  --topic transactions \
#  --bootstrap-server broker:9092 \
#  --partitions 3 \
#  --replication-factor 1

# Verify:

# kafka/bin/kafka-topics.sh --list --bootstrap-server broker:9092

# Paste the output of the --list command here as a comment:
# transactions

In [ ]:
# Part 2: Producer – Generating Transactions
# Task 2.1 – Transaction producer

In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction():
    return {
        "tx_id": f"TX{random.randint(1, 9999):04d}",
        "user_id": f"u{random.randint(1, 20):02d}",
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(["Warsaw", "Krakow", "Gdansk", "Wroclaw"]),
        "category": random.choice(["electronics", "clothing", "food", "books"]),
        "timestamp": datetime.now().isoformat()
    }

for _ in range(50):
    transaction = generate_transaction()

    producer.send('transactions', value=transaction)
    producer.flush()

    print(transaction)

    time.sleep(1)

producer.close()

Writing producer.py


In [ ]:
# Part 3: Stateless Consumer – Filtering
# Task 3.1 – Filter large transactions

In [7]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='filter-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Listening for large transactions (amount > 1000)...")

for message in consumer:
    transaction = message.value

    if transaction["amount"] > 5000:
        print(
            f"ALERT: {transaction['tx_id']} | "
            f"{transaction['amount']:.2f} PLN | "
            f"{transaction['store']} | "
            f"{transaction['category']}"
        )

Overwriting consumer_filter.py


In [ ]:
# Task 3.2 – Transform and enrich

In [3]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='enrich-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    transaction = message.value

    amount = transaction["amount"]

    if amount > 3000:
        transaction["risk_level"] = "HIGH"
    elif amount > 1000:
        transaction["risk_level"] = "MEDIUM"
    else:
        transaction["risk_level"] = "LOW"

    print(transaction)

Writing consumer_enrich.py


In [ ]:
# Part 4: Stateful Consumer – Aggregating
# Task 4.1 – Count transactions per store

In [4]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

for message in consumer:
    transaction = message.value

    store = transaction["store"]
    amount = transaction["amount"]

    # 1. Increment store count
    store_counts[store] += 1

    # 2. Add amount to running total
    total_amount[store] = total_amount.get(store, 0) + amount

    msg_count += 1

    # 3. Print summary every 10 messages
    if msg_count % 10 == 0:
        print("\nStore | Count | Total Amount | Avg Amount")
        print("-" * 50)

        for store in store_counts:
            count = store_counts[store]
            total = total_amount[store]
            avg = total / count

            print(
                f"{store:<10} | "
                f"{count:>5} | "
                f"{total:>12.2f} | "
                f"{avg:>10.2f}"
            )

        print("-" * 50)

Writing consumer_count.py


In [ ]:
# Task 4.2 – Running statistics per category

In [5]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(
    lambda: {
        "count": 0,
        "revenue": 0.0,
        "min_amount": float('inf'),
        "max_amount": float('-inf')
    }
)

msg_count = 0

for message in consumer:
    transaction = message.value

    category = transaction["category"]
    amount = transaction["amount"]

    stats[category]["count"] += 1
    stats[category]["revenue"] += amount
    stats[category]["min_amount"] = min(stats[category]["min_amount"], amount)
    stats[category]["max_amount"] = max(stats[category]["max_amount"], amount)

    msg_count += 1

    if msg_count % 10 == 0:
        print("\nCategory      | Count | Revenue    | Min      | Max")
        print("-" * 60)

        for category, data in stats.items():
            print(
                f"{category:<13} | "
                f"{data['count']:>5} | "
                f"{data['revenue']:>10.2f} | "
                f"{data['min_amount']:>8.2f} | "
                f"{data['max_amount']:>8.2f}"
            )

        print("-" * 60)

Writing consumer_stats.py


In [ ]:
# Part 5: Multiple Consumers
# Task 5.1 – Run everything together
#
# Open 3 terminals in JupyterLab and run simultaneously:
#
# python producer.py (generates events)
# python consumer_filter.py (filters large transactions)
# python consumer_count.py (counts per store)
# Observe how both consumers process the same stream independently.

In [ ]:
# Task 5.2 – Questions

# Answer these questions:
# 1. What happens if you start consumer_filter.py AFTER the producer has finished?
#    (Hint: check auto_offset_reset)
#
# 2. What happens if two consumers have the SAME group_id?
#
# 3. What is the difference between stateless and stateful processing?
#    Give one example of each from this lab.
#
## ANSWERS:
# 1.
# Because auto_offset_reset='earliest', the consumer will read all existing
# messages from the beginning of the topic if the consumer group has never
# consumed them before. Therefore, even if the producer has already finished,
# the consumer can still process all stored transactions.
#
# If auto_offset_reset were set to 'latest', the consumer would only receive
# new messages produced after it starts.
#
# 2.
# Kafka treats them as members of the same consumer group. The topic partitions
# are divided among the consumers, so each message is processed by only one
# consumer in the group.
#
# For example, if the topic has 3 partitions and there are 2 consumers in the
# same group, Kafka may assign 2 partitions to one consumer and 1 partition to
# the other. Together, they share the workload.
#
# 3.
# Stateless processing handles each event independently and does not remember
# previous events.
#
# Example from this lab:
# consumer_filter.py checks whether amount > 1000 and prints an alert.
# Each transaction is processed independently.
#
# Stateful processing keeps information (state) across multiple events and
# updates it as new events arrive.
#
# Example from this lab:
# consumer_count.py maintains running counts and total sales per store.
# Another example is consumer_stats.py, which keeps track of counts,
# revenue, minimum amounts, and maximum amounts for each category.

In [ ]:
# Homework
# Write a consumer that detects velocity anomalies: alert if the same user_id makes more than 3 transactions within 60 seconds. (Hint: keep a dict of {user_id: [timestamps]})
# Push all your code to your Git repository.

In [6]:
%%file velocity_anomalies.py
from kafka import KafkaConsumer
from collections import defaultdict
from datetime import datetime, timedelta
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='velocity-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_transactions = defaultdict(list)

for message in consumer:
    transaction = message.value

    user_id = transaction["user_id"]
    tx_time = datetime.fromisoformat(transaction["timestamp"])

    # Add current transaction time
    user_transactions[user_id].append(tx_time)

    # Keep only transactions from the last 60 seconds
    cutoff = tx_time - timedelta(seconds=60)

    user_transactions[user_id] = [
        t for t in user_transactions[user_id]
        if t >= cutoff
    ]

    # Alert if more than 3 transactions within 60 seconds
    if len(user_transactions[user_id]) > 3:
        print(
            f"VELOCITY ALERT: {user_id} made "
            f"{len(user_transactions[user_id])} transactions "
            f"within 60 seconds"
        )

Writing velocity_anomalies.py
